# Week 1 — Gaussian Process optimisation

Load the original observations and select one new query per function. The notebook is in `Week_01/Python_Files/`, with the initial `.npy` datasets in `Week_01/Data/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

working_dir = Path.cwd()
possible_data_folders = [working_dir.parent / "Data", working_dir / "Data"]
data_folder = next((folder for folder in possible_data_folders if folder.is_dir()), None)

if data_folder is None:
    raise FileNotFoundError(
        f"Cannot find Week_01/Data from {working_dir}. "
        "Run this notebook from Week_01/Python_Files or Week_01."
    )

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Load and inspect the initial observations

In [2]:
data = {}

for i in range(1, 9):
    
    function_folder = data_folder / f"function_{i}"
    
    X = np.load(function_folder / "initial_inputs.npy")
    y = np.load(function_folder / "initial_outputs.npy")
    
    columns = [f"x{j+1}" for j in range(X.shape[1])]
    
    df = pd.DataFrame(X, columns=columns)
    df["output"] = y
    
    data[i] = df
    
    highest_row = df.loc[df["output"].idxmax()]
    lowest_row = df.loc[df["output"].idxmin()]
    
    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"Number of observations: {len(df)}")
    print(f"Number of dimensions: {X.shape[1]}")
    
    print("\nHighest observed output:")
    print(highest_row)
    
    print("\nLowest observed output:")
    print(lowest_row)
    
    print(
        f"\nOutput range: "
        f"{df['output'].min():.6e} to "
        f"{df['output'].max():.6e}"
    )
    
    print("\n")


FUNCTION 1
Number of observations: 10
Number of dimensions: 2

Highest observed output:
x1        7.310236e-01
x2        7.329999e-01
output    7.710875e-16
Name: 2, dtype: float64

Lowest observed output:
x1        0.650114
x2        0.681526
output   -0.003606
Name: 4, dtype: float64

Output range: -3.606063e-03 to 7.710875e-16


FUNCTION 2
Number of observations: 10
Number of dimensions: 2

Highest observed output:
x1        0.702637
x2        0.926564
output    0.611205
Name: 9, dtype: float64

Lowest observed output:
x1        0.142699
x2        0.349005
output   -0.065624
Name: 2, dtype: float64

Output range: -6.562362e-02 to 6.112052e-01


FUNCTION 3
Number of observations: 15
Number of dimensions: 3

Highest observed output:
x1        0.492581
x2        0.611593
x3        0.340176
output   -0.034835
Name: 3, dtype: float64

Lowest observed output:
x1        0.151837
x2        0.439991
x3        0.990882
output   -0.398926
Name: 6, dtype: float64

Output range: -3.989255e-01 to

## 2. Fit a Gaussian Process for each function

In [3]:
gps = {}

for i in range(1, 9):
    
    df = data[i]
    
    X = df.drop(columns="output").values
    y = df["output"].values
    
    kernel = RBF(length_scale=1.0)
    
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        normalize_y=True,
        random_state=42
    )
    
    gp.fit(X, y)
    
    gps[i] = gp
    
    print(f"Function {i}")
    print(f"Fitted kernel: {gp.kernel_}")
    print()


Function 1
Fitted kernel: RBF(length_scale=1e-05)

Function 2
Fitted kernel: RBF(length_scale=0.125)

Function 3
Fitted kernel: RBF(length_scale=0.188)

Function 4
Fitted kernel: RBF(length_scale=0.5)

Function 5
Fitted kernel: RBF(length_scale=1e-05)

Function 6
Fitted kernel: RBF(length_scale=1e-05)

Function 7
Fitted kernel: RBF(length_scale=0.281)

Function 8
Fitted kernel: RBF(length_scale=0.892)



## 3. Investigate the convergence warnings

In [4]:
warning_functions = [1, 5, 6]

for i in warning_functions:

    df = data[i]

    X = df.drop(columns="output").values
    y = df["output"].values

    # Allow the optimiser to search below the previous 1e-5 lower bound
    kernel = RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-10, 1e5)
    )

    gp_check = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        normalize_y=True,
        random_state=42
    )

    gp_check.fit(X, y)

    print(f"Function {i}")
    print("New fitted kernel:", gp_check.kernel_)
    print()


Function 1
New fitted kernel: RBF(length_scale=1e-10)

Function 5
New fitted kernel: RBF(length_scale=0.269)

Function 6
New fitted kernel: RBF(length_scale=1e-10)



## 4. Refit Function 5 with wider kernel bounds

In [5]:
df = data[5]

X = df.drop(columns="output").values
y = df["output"].values

kernel = RBF(
    length_scale=1.0,
    length_scale_bounds=(1e-10, 1e5)
)

gp5 = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)

gp5.fit(X, y)

gps[5] = gp5

print("Function 5 fitted kernel:", gps[5].kernel_)


Function 5 fitted kernel: RBF(length_scale=0.269)


## 5. Generate candidate points and estimate their outputs

In [6]:
rng = np.random.default_rng(42)

n_candidates = 100000

candidates = {}
predicted_means = {}
predicted_stds = {}

for i in range(1, 9):

    df = data[i]

    n_dimensions = len(df.columns) - 1

    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    mean, std = gps[i].predict(
        X_candidates,
        return_std=True
    )

    candidates[i] = X_candidates
    predicted_means[i] = mean
    predicted_stds[i] = std

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Dimensions:", n_dimensions)
    print("Candidate points:", len(X_candidates))

    print(
        "Predicted output range:",
        f"{mean.min():.6f}",
        "to",
        f"{mean.max():.6f}"
    )

    print(
        "Uncertainty range:",
        f"{std.min():.6f}",
        "to",
        f"{std.max():.6f}"
    )

    print()


FUNCTION 1
Dimensions: 2
Candidate points: 100000
Predicted output range: -0.000361 to -0.000361
Uncertainty range: 0.001082 to 0.001082

FUNCTION 2
Dimensions: 2
Candidate points: 100000
Predicted output range: -0.077317 to 0.643977
Uncertainty range: 0.000068 to 0.225365

FUNCTION 3
Dimensions: 3
Candidate points: 100000
Predicted output range: -0.397833 to -0.018176
Uncertainty range: 0.002777 to 0.084213

FUNCTION 4
Dimensions: 4
Candidate points: 100000
Predicted output range: -35.682037 to -0.922445
Uncertainty range: 0.067992 to 5.263611

FUNCTION 5
Dimensions: 4
Candidate points: 100000
Predicted output range: -71.908033 to 1113.402974
Uncertainty range: 12.509737 to 245.522395

FUNCTION 6
Dimensions: 5
Candidate points: 100000
Predicted output range: -1.495390 to -1.495390
Uncertainty range: 0.449000 to 0.449000

FUNCTION 7
Dimensions: 6
Candidate points: 100000
Predicted output range: -0.020560 to 1.208211
Uncertainty range: 0.061869 to 0.302129

FUNCTION 8
Dimensions: 8
Cand

## 6. Rank candidates using Upper Confidence Bound

In [7]:
kappa = 2.5

ucb_values = {}

for i in range(1, 9):

    mean = predicted_means[i]
    std = predicted_stds[i]

    ucb = mean + kappa * std
    ucb_values[i] = ucb

    best_index = np.argmax(ucb)

    best_candidate = candidates[i][best_index]
    best_mean = mean[best_index]
    best_std = std[best_index]
    best_ucb = ucb[best_index]

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:", best_candidate)
    print("Predicted output:", best_mean)
    print("Uncertainty:", best_std)
    print("UCB:", best_ucb)
    print()


FUNCTION 1
Selected point: [0.77395605 0.43887844]
Predicted output: -0.00036060626443627054
Uncertainty: 0.0010818187933090686
UCB: 0.002343940718836401

FUNCTION 2
Selected point: [0.8217782  0.98030138]
Predicted output: 0.5373024810236847
Uncertainty: 0.17485188463025975
UCB: 0.9744321925993341

FUNCTION 3
Selected point: [0.40949935 0.39899996 0.51948667]
Predicted output: -0.04187059316125945
Uncertainty: 0.07197438103562513
UCB: 0.1380653594278034

FUNCTION 4
Selected point: [0.39915655 0.43320344 0.35769894 0.461233  ]
Predicted output: -1.109357344181845
Uncertainty: 0.9929487520649243
UCB: 1.3730145359804657

FUNCTION 5
Selected point: [0.35226763 0.85495745 0.92921014 0.99954104]
Predicted output: 1003.7100993151818
Uncertainty: 139.19689039784117
UCB: 1351.7023253097846

FUNCTION 6
Selected point: [0.42871117 0.38510403 0.39053256 0.72350556 0.95165422]
Predicted output: -1.4953899396939465
Uncertainty: 0.44899959039781934
UCB: -0.3728909636993982

FUNCTION 7
Selected point

## 7. Explore less-sampled areas for Functions 1 and 6

In [8]:
exploration_points = {}

for i in [1, 6]:

    df = data[i]

    X_observed = df.drop(columns="output").values

    X_candidates = candidates[i]

    distances = cdist(X_candidates, X_observed)

    nearest_distance = distances.min(axis=1)

    most_unexplored_index = np.argmax(nearest_distance)

    most_unexplored_point = X_candidates[most_unexplored_index]

    exploration_points[i] = most_unexplored_point

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected exploration point:")
    print(most_unexplored_point)

    print(
        "Distance to nearest existing observation:",
        nearest_distance[most_unexplored_index]
    )

    print()


FUNCTION 1
Selected exploration point:
[0.00124268 0.99691888]
Distance to nearest existing observation: 0.39492227534104346

FUNCTION 6
Selected exploration point:
[0.91049992 0.03922596 0.99959186 0.08178911 0.93948142]
Distance to nearest existing observation: 0.8860560302850456



## 8. Select and compare the final query points

In [9]:
final_points = {}

# Functions 1 and 6 use our exploration points
final_points[1] = exploration_points[1]
final_points[6] = exploration_points[6]

for i in [2, 3, 4, 5, 7, 8]:
    
    best_index = np.argmax(ucb_values[i])
    final_points[i] = candidates[i][best_index]


for i in range(1, 9):
    
    df = data[i]
    
    best_row = df.loc[df["output"].idxmax()]
    best_X = best_row.drop("output").values.astype(float)
    best_y = best_row["output"]
    
    new_X = final_points[i]
    
    distance = np.linalg.norm(new_X - best_X)
    
    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    
    print("Current best point:")
    print(np.round(best_X, 6))
    
    print("Current best output:")
    print(best_y)
    
    print("\nProposed new point:")
    print(np.round(new_X, 6))
    
    print("\nDistance from current best:")
    print(round(distance, 4))
    
    print()


FUNCTION 1
Current best point:
[0.731024 0.733   ]
Current best output:
7.710875114502849e-16

Proposed new point:
[0.001243 0.996919]

Distance from current best:
0.776

FUNCTION 2
Current best point:
[0.702637 0.926564]
Current best output:
0.6112052157614438

Proposed new point:
[0.821778 0.980301]

Distance from current best:
0.1307

FUNCTION 3
Current best point:
[0.492581 0.611593 0.340176]
Current best output:
-0.034835313350078584

Proposed new point:
[0.409499 0.399    0.519487]

Distance from current best:
0.2903

FUNCTION 4
Current best point:
[0.577766 0.428772 0.425826 0.249007]
Current best output:
-4.025542281908162

Proposed new point:
[0.399157 0.433203 0.357699 0.461233]

Distance from current best:
0.2857

FUNCTION 5
Current best point:
[0.224189 0.84648  0.879484 0.878516]
Current best output:
1088.8596181962705

Proposed new point:
[0.352268 0.854957 0.92921  0.999541]

Distance from current best:
0.1833

FUNCTION 6
Current best point:
[0.728186 0.154693 0.732552 0

### Notes

The candidate search uses a fixed random seed of 42. The model uses 100,000 candidate points per function and an Upper Confidence Bound exploration parameter of 2.5. The final candidate coordinates can vary slightly with library versions or numerical optimisation settings.

## 9. Final submission summary

The table below records the selected method, candidate coordinates and distance from the best initial observation. These are proposed queries; their returned objective values are assessed in the following week.

In [10]:
summary_rows = []
for i in range(1, 9):
    df = data[i]
    best_row = df.loc[df["output"].idxmax()]
    best_x = best_row.drop("output").to_numpy(dtype=float)
    point = np.asarray(final_points[i], dtype=float)
    summary_rows.append({
        "Function": i,
        "Method": "Max-distance exploration" if i in (1, 6) else "GP + UCB",
        "Best initial output": float(best_row["output"]),
        "Distance from best": float(np.linalg.norm(point - best_x)),
        "Submission coordinates": "-".join(f"{v:.6f}" for v in point)
    })

submission_summary = pd.DataFrame(summary_rows)
print("WEEK 1 — FINAL PROPOSED QUERIES")
print(submission_summary.to_string(index=False))


WEEK 1 — FINAL PROPOSED QUERIES
 Function                   Method  Best initial output  Distance from best                                                  Submission coordinates
        1 Max-distance exploration         7.710875e-16            0.776037                                                       0.001243-0.996919
        2                 GP + UCB         6.112052e-01            0.130700                                                       0.821778-0.980301
        3                 GP + UCB        -3.483531e-02            0.290260                                              0.409499-0.399000-0.519487
        4                 GP + UCB        -4.025542e+00            0.285660                                     0.399157-0.433203-0.357699-0.461233
        5                 GP + UCB         1.088860e+03            0.183292                                     0.352268-0.854957-0.929210-0.999541
        6 Max-distance exploration        -7.142649e-01            1.128056     